In [ ]:
import time


start_time = time.time()

# 0. Загрузка датасета


In [ ]:
# !gdown 1sbnKeVmiOmT4ZZHIbXIxuBWF9CQLWt6s
# from IPython.display import clear_output
# !unzip week-5-competition-geo-satellites-classification.zip
# clear_output()

## 1. Подготовка
Укажите в `DATA_DIR` папку, в которой лежат `train/`, `val/`, `test/` и CSV-файлы
(на Kaggle: `/kaggle/input/<название-соревнования>`).

**Воспроизводимость (правило соревнования):** держите `SEED` фиксированным и вызывайте `seed_everything()` перед
обучением, чтобы повторный запуск ноутбука давал тот же результат.

In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import v2

DATA_DIR = Path('/kaggle/input/datasets/oripyro/week-5')
# DATA_DIR = Path("week-5-competition-geo-satellites-classification") # Локальный путь в Windows
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

def seed_everything(seed=None):
    seed = SEED if seed is None else seed               # SEED задаётся в блоке гиперпараметров
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

print("device:", DEVICE)
print("data folder:", sorted(p.name for p in DATA_DIR.iterdir()))

## 1.1 Гиперпараметры
Все числовые гиперпараметры, предобученная сеть и классификатор собраны здесь. Остальной код берёт значения отсюда.

In [ ]:
# ===================== ГИПЕРПАРАМЕТРЫ =====================

# --- Общие ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_THREADS = 16                  # torch.set_num_threads

# --- Изображения и нормализация ---
IMG_SIZE = 224                    # исходный размер, чтобы вытянуть максимум информации
MEAN = (0.485, 0.456, 0.406)      # нормализация ImageNet
STD  = (0.229, 0.224, 0.225)

# --- Аугментации (train_transform) ---
HFLIP_P = 0.5                     # отражение слева направо
VFLIP_P = 0.5                     # отражение сверху вниз
ROTATION_ANGLES = (0, 90, 180, 270)
BRIGHTNESS = 0.35                 # ColorJitter: ×(1-b) ... ×(1+b)
CONTRAST = 0.35
BLUR_KERNEL = 9
BLUR_SIGMA = (0.5, 3.0)
BLUR_P = 0.3
JPEG_QUALITY = (10, 80)
JPEG_P = 0.5
NOISE_SIGMAS = (0.02, 0.05, 0.1, 0.15, 0.2)
NOISE_P = 0.1
GRAY_P = 0.1

# --- Обучение ---
BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-4
LOG_EVERY = 1                    # печатать метрики каждые N эпох

seed_everything()                 # фиксируем сид до инициализации весов CLASSIFIER

# --- Предобученная сеть ---
BACKBONE = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
NUM_FEATURES = BACKBONE.fc.in_features    # 512 для resnet18, 2048 для resnet50
UNFREEZE_LAYERS = []                     # например ["layer3", "layer4"]; [] — всё заморожено

# --- Классификатор ---
NUM_CLASSES = 20

CLASSIFIER = nn.Sequential(
    nn.BatchNorm1d(NUM_FEATURES),
    nn.Dropout(0.3),
    nn.Linear(NUM_FEATURES, NUM_CLASSES),
)

In [ ]:
# torch.set_num_threads(NUM_THREADS) # Больше ядер
print(torch.get_num_threads())
print(DEVICE)

In [ ]:
train_table = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
val_table = pd.read_csv(DATA_DIR / "val.csv", dtype={"id": str})
test_table = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
classes = pd.read_csv(DATA_DIR / "classes.csv").sort_values("label").class_name.tolist()
assert NUM_CLASSES == len(classes), f"NUM_CLASSES={NUM_CLASSES}, а в classes.csv {len(classes)} классов"

print(f"train: {len(train_table)} images | val: {len(val_table)} | test: {len(test_table)} | classes: {NUM_CLASSES}")
print(classes)
train_table.head()

## 3. Преобразования и аугментации с `transforms.Compose`

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=HFLIP_P),                         # отражение слева направо
    transforms.RandomVerticalFlip(p=VFLIP_P),                           # отражение сверху вниз
    transforms.RandomChoice([transforms.RandomRotation((angle, angle))  # поворот на один из ROTATION_ANGLES
                             for angle in ROTATION_ANGLES]),
    transforms.ColorJitter(brightness=BRIGHTNESS, contrast=CONTRAST),   # яркость и контраст
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA)], p=BLUR_P),  # размытие
    transforms.RandomApply([v2.JPEG(quality=JPEG_QUALITY)], p=JPEG_P),  # JPEG-сжатие
    transforms.RandomApply([v2.Compose([                                # шум
                                v2.ToImage(),
                                v2.ToDtype(torch.float32, scale=True),
                                v2.RandomChoice([v2.GaussianNoise(sigma=s) for s in NOISE_SIGMAS]),
                                v2.ToPILImage(),
                            ]),], p=NOISE_P),
    transforms.RandomGrayscale(p=GRAY_P),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([         # валидация и тест: помните, никаких аугментаций для этих частей!!!
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## 3.1 Проверка аугментации

In [ ]:
def image_file(folder, image_id):
    """Путь к изображению. Id — это имена файлов вроде '1000.jpg' (просто числа тоже подходят)."""
    name = image_id if image_id.endswith(".jpg") else f"{image_id}.jpg"
    return DATA_DIR / folder / name

img = Image.open(image_file("train", "1.jpg")).convert("RGB")

def show_transforms(image_id, tfms, folder="train", size=3):
    """Оригинал + результат каждой трансформации из словаря {название: transform}."""
    img = Image.open(image_file(folder, image_id)).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BOX)
    fig, axes = plt.subplots(1, len(tfms) + 1, figsize=(size * (len(tfms) + 1), size))
    axes[0].imshow(img); axes[0].set_title("original", fontsize=9)
    for ax, (name, t) in zip(axes[1:], tfms.items()):
        ax.imshow(t(img)); ax.set_title(name, fontsize=9)
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

show_transforms("10.jpg", {
    "blur σ=1":     transforms.GaussianBlur(9, sigma=1.0),
    "blur σ=3":     transforms.GaussianBlur(15, sigma=3.0),
    "invert":       transforms.RandomInvert(p=1.0),
    "sharpness":    transforms.RandomAdjustSharpness(12, p=1.0),
    "jitter":       transforms.ColorJitter(brightness=0.5, contrast=0.5),
    "JPEG":         v2.JPEG(quality=8),
    "noise":        v2.Compose([
                        v2.ToImage(),
                        v2.ToDtype(torch.float32, scale=True),
                        v2.GaussianNoise(sigma=0.02),
                        v2.ToPILImage(),
                    ]),
})

## 4. `Dataset`, читающий изображения из папок

In [ ]:
class ContestDataset(Dataset):
    def __init__(self, table, folder, transform, img_size=IMG_SIZE):
        self.ids = table.id.tolist()
        self.labels = table.label.tolist() if "label" in table.columns else [-1] * len(table)
        self.transform = transform
        self.images = []
        for image_id in tqdm(self.ids, desc=f"loading {folder}"):
            with Image.open(image_file(folder, image_id)) as img:
                self.images.append(img.convert("RGB").resize((img_size, img_size), Image.BOX))

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]

train_set = ContestDataset(train_table, "train", train_transform)
val_set = ContestDataset(val_table, "val", eval_transform)
test_set = ContestDataset(test_table, "test", eval_transform)

image, label = train_set[0]
print("one sample:", tuple(image.shape), "| label:", label, f"({classes[label]})")

## 5. Функции обучения и оценки
`DataLoader` группирует объекты в батчи (и перемешивает обучающую выборку). Одни и те же функции используются для всех
моделей ноутбука, включая предобученные в задании:

* `evaluate` считает loss и accuracy на наборе данных (без обучения и без аугментаций);
* `predict` возвращает предсказанный класс для каждого изображения набора;
* `train_model` обучает заданное число эпох. После каждой эпохи записывает **loss** и **accuracy** на обучающей и
  валидационной выборках, а в конце возвращается к весам **лучшей эпохи** (с наибольшей валидационной accuracy).
* `make_loader` помогает сохранять порядок изображений ради воспроизводимости.

Модели возвращают сырые оценки (логиты); `nn.CrossEntropyLoss` применяет softmax внутри себя.

In [ ]:
criterion = nn.CrossEntropyLoss()

TRAIN_GENERATOR = torch.Generator().manual_seed(SEED)   # создаётся один раз

def make_loader(dataset, shuffle):
    return DataLoader(
        dataset, batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=TRAIN_GENERATOR if shuffle else None,
        num_workers=4, pin_memory=True,
        persistent_workers=shuffle,
    )

@torch.no_grad()
def evaluate(model, dataset):
    model.eval()
    total_loss, correct = 0.0, 0
    for images, labels in make_loader(dataset, shuffle=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item() * len(images)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(dataset), correct / len(dataset)

@torch.no_grad()
def predict(model, dataset):
    model.eval()
    predictions = []
    for images, _ in make_loader(dataset, shuffle=False):
        outputs = model(images.to(DEVICE))
        predictions.append(outputs.argmax(dim=1).cpu())
    return torch.cat(predictions).numpy()

def train_model(model, name, train_data, val_data, epochs=EPOCHS, lr=LR):
    model = model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=lr)
    train_loader = make_loader(train_data, shuffle=True)

    history, best_acc, best_weights = [], -1.0, None    # -1.0 — просто число для сравнения
    for epoch in range(1, epochs + 1):

        # шаг обучения (тот же, что мы изучали для более простых моделей)
        model.train()
        total_loss, correct = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_loss, train_acc = total_loss / len(train_data), correct / len(train_data)

        # шаг валидации
        val_loss, val_acc = evaluate(model, val_data)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "train_acc": train_acc, "val_acc": val_acc})

        if val_acc > best_acc:                                 # запоминаем лучшую эпоху и её веса
            best_acc, best_weights = val_acc, copy.deepcopy(model.state_dict())
        if epoch == 1 or epoch % LOG_EVERY == 0:
            print(f"{name} | epoch {epoch:2d} | train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
                  f"train acc {train_acc:.3f} | val acc {val_acc:.3f}")

    model.load_state_dict(best_weights)                        # возвращаемся к лучшей эпохе
    print(f"{name}: best validation accuracy {best_acc:.3f}")
    return pd.DataFrame(history)

def plot_history(history, name):
    """Loss (ошибка) и accuracy по эпохам, для обучающей и валидационной выборок."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.epoch, history.train_loss, label="training")
    axes[0].plot(history.epoch, history.val_loss, label="validation")
    axes[0].set_title(f"{name}: loss (error)"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(history.epoch, history.train_acc, label="training")
    axes[1].plot(history.epoch, history.val_acc, label="validation")
    axes[1].set_title(f"{name}: accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Обучаем модель

In [ ]:
backbone = copy.deepcopy(BACKBONE)
backbone.fc = nn.Identity()

seed_everything()

for param in backbone.parameters():
    param.requires_grad = False

classifier = copy.deepcopy(CLASSIFIER)          # копия, чтобы обучение не меняло CLASSIFIER из блока
custom_1 = nn.Sequential(backbone, classifier)

for layer_name in ["layer3", "layer4"]:
    for param in getattr(custom_1[0], layer_name).parameters():
        param.requires_grad = True
print("trainable parameters:", count_parameters(custom_1), "of", sum(p.numel() for p in custom_1.parameters()))
history_1_step = train_model(custom_1, "layer3, layer4, classifier", train_set, val_set, epochs=5, lr=1e-3)

for layer_name in ["layer3"]:
    for param in getattr(custom_1[0], layer_name).parameters():
        param.requires_grad = False
print("trainable parameters:", count_parameters(custom_1), "of", sum(p.numel() for p in custom_1.parameters()))
history_2_step = train_model(custom_1, "layer4, classifier", train_set, val_set, epochs=5, lr=1e-3)

for layer_name in ["layer4"]:
    for param in getattr(custom_1[0], layer_name).parameters():
        param.requires_grad = False
print("trainable parameters:", count_parameters(custom_1), "of", sum(p.numel() for p in custom_1.parameters()))
history_3_step = train_model(custom_1, "classifier", train_set, val_set, epochs=10, lr=1e-4)


In [ ]:
# Проверка генератора загрузчика: подряд идущие вызовы дают разный порядок, сброс воспроизводит его
def first_batch_labels():
    _, labels = next(iter(make_loader(train_set, shuffle=True)))
    return labels

TRAIN_GENERATOR.manual_seed(SEED)
labels_first = first_batch_labels()
labels_second = first_batch_labels()
TRAIN_GENERATOR.manual_seed(SEED)
labels_again = first_batch_labels()

print("1-й и 2-й вызовы различаются:", not torch.equal(labels_first, labels_second))
print("после сброса совпадает с 1-м:", torch.equal(labels_first, labels_again))
assert not torch.equal(labels_first, labels_second)
assert torch.equal(labels_first, labels_again)

In [ ]:
# Скачиваем результат
submission = pd.DataFrame({"id": test_set.ids, "label": predict(custom_1, test_set)})

# проверка формата перед загрузкой
assert list(submission.columns) == ["id", "label"]
assert submission.id.tolist() == test_table.id.tolist()
assert submission.label.between(0, NUM_CLASSES - 1).all()

submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
plot_history(history_1_step, "layer3, layer4, classifier")
plot_history(history_2_step, "layer4, classifier")
plot_history(history_3_step, "classifier")


In [ ]:
end_time = time.time()
elapsed_minutes = (end_time - start_time) / 60

print(f"Elapsed time: {elapsed_minutes:.2f} minutes")